In [23]:
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

In [24]:
## load the trained model, scaler, and label encoder
model=load_model('churn_model.h5')

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

In [25]:
## example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [26]:
## One hot encode geography

geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

d:\Data Science\ANN_Project\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [27]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [29]:
input_df = pd.DataFrame([input_data])
input_df = pd.concat([input_df, geo_encoded_df], axis=1).drop(columns=['Geography'], errors='ignore')
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [30]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [31]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [32]:
## scaling the input data
scaled_input = scaler.transform(input_df)

In [34]:
scaled_input

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [37]:
## predicting churn
prediction = model.predict(scaled_input)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step


array([[0.03313731]], dtype=float32)

In [38]:
pred_prob = prediction[0][0]
pred_prob

np.float32(0.03313731)

In [39]:
if pred_prob > 0.5:
    print(f"The customer is likely to churn with a probability of {pred_prob:.2f}.")    
else:
    print(f"The customer is unlikely to churn with a probability of {pred_prob:.2f}.")

The customer is unlikely to churn with a probability of 0.03.
